# Simulating randomized benchmarking with Quax

Randomized benchmarking is a common benchmark for determining the fidelity of gates. 

This notebook demonstrates how we can construct and simulate randomized benchmarking sequences using quax.

It demonstrates how the basic tools of batch operations, gradients and superoperators can be used to simulate and interpret the behaviour of gates.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import jax
import jax.numpy as jnp
import quax as qx

# enable 64 bit
jax.config.update("jax_enable_x64", True)

## Parameters

In [ ]:
num_randomizations = 30
depths = [2, 4, 8, 16, 32, 64, 128, 256]
seed = 485
t1 = 30.0  # us
t2 = 20.0  # us
gate_time = 50e-3  # us

Generate the noisy superoperators

In [ ]:
t1s = jnp.array([t1])
t2s = jnp.array([t2])
gate_time = gate_time

### Generate the RB sequences

First, let's generate the sets of random 1Q Clifford gates and the inversions

In [ ]:
key = jax.random.key(seed)

num_cliffords = qx.ensembles.CLIFFORDS_1Q.ensemble_size[0]


def invert_sequence(random_unitaries: qx.Unitary) -> qx.Unitary:
    """
    Given a sequence of 1Q unitaries, compute the overall unitary and invert it.

    :param random_unitaries: Ensemble of random sequences dimension (num_randomizations, depth, 2, 2).
    :return: (num_randomizations, depth + 1, 2, 2) ensemble which accumuluate to the identity.
    """

    def thunk(carry, u):
        return u @ carry, None

    # compute the reduced process to find the inversion clifford
    identity = qx.Unitary(
        jnp.broadcast_to(qx.gates.I.data, (num_randomizations,) + qx.gates.I.data.shape), num_qubits=1
    )
    reduced_process, _ = jax.lax.scan(thunk, identity, random_unitaries)
    inversion = reduced_process.h

    # append the inversion unitary
    random_unitaries = qx.Unitary(
        data=jnp.concatenate([random_unitaries.data, inversion.data[jnp.newaxis, :, :, :]], axis=0),
        num_qubits=1,
    )

    return random_unitaries


# generate a set of Cliffords (num_randomizations x len(depths))
cliffords = []
for depth in depths:
    subkey, key = jax.random.split(key)
    rand_ints = jax.random.randint(
        subkey,
        shape=(depth - 1, num_randomizations),
        minval=0,
        maxval=num_cliffords,
    )
    # sample the random cliffords
    random_cliffords = qx.Unitary(data=qx.ensembles.CLIFFORDS_1Q.data[rand_ints], num_qubits=1)

    # invert the sequence and append
    random_cliffords = invert_sequence(random_cliffords)

    # collect the full sequence
    cliffords.append(random_cliffords)

Compute the final states

In [ ]:
def compute_final_state(random_sequences: qx.Unitary) -> qx.StateVector:
    """
    Compute the final state of an ensemble of random sequences.
    Will reduce on the leading dimension of the ensemble, so the input should be (depth, ..., 2, 2).

    :param random_sequences: (depth, ..., 2, 2) ensemble of random sequences.
    :return: (...) final state vector after applying the sequence.
    """
    ensemble_size = random_sequences.ensemble_size[1:]
    initial_state = qx.zero_state_vector(1, ensemble_size)

    def thunk(state, clifford):
        new_state = clifford @ state
        return new_state, None

    final_state, _ = jax.lax.scan(thunk, initial_state, random_sequences)
    return final_state


for i, depth_cliffords in enumerate(cliffords):
    final_state = compute_final_state(depth_cliffords)
    zero_probs = qx.bitstring_probability(final_state, jnp.array([0]))
    print(f"Probability of measuring '0' at the end of sequences for depth {depths[i]}: {zero_probs.mean():.4f}")

### Decompose to ZXZXZ

The sequence of gates which are `(depth, num_randomizations, 2, 2)` can be decomposed to native unitary operations. This yields a unitary array of dimension `(depth, num_randomizations, 5, 2, 2)`.

We perform the decomposition and see that the final states are still the 0 bitstring.

In [ ]:
def to_zxzxz(random_sequence: qx.Unitary) -> qx.Unitary:
    """
    Convert a sequence of 1Q Cliffords to a sequence of ZXZXZ gates.

    :param random_sequence: (depth, num_randomizations, 2, 2) ensemble of random sequences.
    :return: (depth*5, num_randomizations, 2, 2) ensemble of ZXZXZ angles.
    """
    depth, num_randomizations = random_sequence.ensemble_size
    angles = qx.to_zxzxz_angles(random_sequence)  # (depth, num_randomizations, 3)
    rz = qx.gates.RZ(angles)  # (depth, num_randomizations, 3, 2, 2)
    sx = jnp.broadcast_to(
        qx.gates.RX(jnp.pi / 2).data, (depth, num_randomizations, 2, 2)
    )  # (depth, num_randomizations, 2, 2)
    # finally, we need to interleave the rz and sx gates to get the full sequence of ZXZXZ gates
    # following the pattern RZ[0, :, 0], SX, RZ[0, :, 1], SX, RZ[0, :, 2]
    block = jnp.stack(
        [rz.data[:, :, 0], sx, rz.data[:, :, 1], sx, rz.data[:, :, 2]],
        axis=1,
    )  # (depth, num_randomizations, 5, 2, 2)

    tensor = block.reshape(depth * 5, num_randomizations, 2, 2)

    return qx.Unitary(tensor, num_qubits=1)


zxzxz_cliffords = []
for i, depth_cliffords in enumerate(cliffords):
    native_sequence = to_zxzxz(depth_cliffords)

    final_state = compute_final_state(native_sequence)
    zero_probs = qx.bitstring_probability(final_state, jnp.array([0]))
    print(f"Probability of measuring '0' at the end of sequences for depth {depths[i]}: {zero_probs.mean():.4f}")

    zxzxz_cliffords.append(native_sequence)

### Adding noise

We wish to add noise into the equation. For this, we'll need to promote our unitaries to superoperators.

In [ ]:
def to_noisy_zxzxz(random_sequence: qx.Unitary, sx_channel: qx.SuperOp) -> qx.SuperOp:
    """
    Convert a sequence of 1Q Cliffords to a sequence of ZXZXZ superoperators.

    The SX gates will be the provided channel, while the RZ gates will be ideal unitaries.

    :param random_sequence: (depth, num_randomizations, 2, 2) ensemble of random sequences.
    :param sx_channel: (2, 2) superoperator representing the SX gate with noise.
    :return: (depth*5, num_randomizations, 2, 2) ensemble of ZXZXZ superoperators.
    """
    depth, num_randomizations = random_sequence.ensemble_size
    angles = qx.to_zxzxz_angles(random_sequence)  # (depth, num_randomizations, 3)
    rz = qx.unitary_to_superop(qx.gates.RZ(angles))  # (depth, num_randomizations, 3, *dims)
    sx = jnp.broadcast_to(
        sx_channel.data, (depth, num_randomizations) + sx_channel.data.shape
    )  # (depth, num_randomizations, *dims)
    # interleave following the pattern RZ[0, :, 0], SX, RZ[0, :, 1], SX, RZ[0, :, 2]
    block = jnp.stack(
        [rz.data[:, :, 0], sx, rz.data[:, :, 1], sx, rz.data[:, :, 2]],
        axis=1,
    )  # (depth, num_randomizations, 5, *dims)

    tensor = block.reshape(depth * 5, num_randomizations, *block.shape[3:])

    return qx.SuperOp(tensor, num_qubits=1)


def compute_final_matrix(random_sequences: qx.SuperOp) -> qx.DensityMatrix:
    """
    Compute the final state of an ensemble of random sequences.
    Will reduce on the leading dimension of the ensemble, so the input should be (depth, ..., 2, 2, 2, 2).

    :param random_sequences: (depth, ..., 2, 2, 2, 2) ensemble of random sequences.
    :return: (...) final state vector after applying the sequence.
    """
    ensemble_size = random_sequences.ensemble_size[1:]
    initial_state = qx.zero_state_matrix(1, ensemble_size)

    def thunk(state, clifford):
        new_state = clifford @ state
        return new_state, None

    final_state, _ = jax.lax.scan(thunk, initial_state, random_sequences)
    return final_state

Next, we'll need to generate a channel. We'll create a simple thermal relaxation channel here using the Lindbladian.

In [ ]:
sx_superop = qx.choi_to_superop(qx.thermal_relaxation_choi(t1s, t2s, gate_time)) @ qx.gates.RX(jnp.pi / 2)

We can once again compute the probabiltiy of measuring 0. While with the unitary operators, the probability of measuring 0 was always 1.0, with the noisy gates, we'll find that the probability decreases with depth.

In [ ]:
for i, depth_cliffords in enumerate(cliffords):
    native_sequence = to_noisy_zxzxz(depth_cliffords, sx_superop)
    final_state = compute_final_matrix(native_sequence)
    zero_probs = qx.bitstring_probability(final_state, jnp.array([0]))
    print(f"Probability of measuring '0' at the end of sequences for depth {depths[i]}: {zero_probs.mean():.4f}")

## Adding the third state

Ideally, superconducting qubits are two-state systems, but in reality they have many excited states. Sometimes, a qubit can become excited to the $|2\rangle$ state. We can model this by promoting our superoperators from qubit superoperators to qutrit superoperators. We'll do this, and set up a simple channel which has some leakage probability as well as a seepage probability.

We can repeat the experiment, but this time we'll observe the probability of measuring 0, 1 and 2.

In [ ]:
# Skip for now
# TODO: Qutrit support will be added with #20

### Determining the sensitivity

Jax enables the simple determination of gradients. This is useful in many scenarios, but here we will use it to determine the sensitivity of the bitstring probability to the noise parameters. We'll compute the gradient of the output bitstring with respect to each parameter, learning which how much each sort of error affects the overall fidelity.

In [ ]:
def compute_probabilities(random_sequences: qx.SuperOp) -> jax.Array:
    """
    Compute the probability of measuring '0' for the sequence.

    :param random_sequences: (depth, ..., 2, 2, 2, 2) ensemble of random sequences.
    :return: (...) probabilities.
    """
    final_state = compute_final_matrix(random_sequences)
    return qx.bitstring_probability(final_state, jnp.array([0]))


def probability_ensemble(t1_val, t2_val, cliffords_list):
    """
    Compute the mean probability of measuring |0> across all depths.

    :param t1_val: T1 relaxation time (scalar).
    :param t2_val: T2 relaxation time (scalar).
    :param cliffords_list: List of Clifford sequences for each depth.
    :return: Array of 0 probabilities, num_randomizations per depth.
    """
    t1_arr = jnp.array([t1_val])
    t2_arr = jnp.array([t2_val])
    sx = qx.choi_to_superop(qx.thermal_relaxation_choi(t1_arr, t2_arr, gate_time)) @ qx.gates.RX(jnp.pi / 2)

    probs = []
    for depth_cliffords in cliffords_list:
        native_sequence = to_noisy_zxzxz(depth_cliffords, sx)
        zero_probs = compute_probabilities(native_sequence)
        probs.append(zero_probs)

    return jnp.stack(probs)


# Compute the Jacobian of the mean probabilities with respect to t1 and t2
jacobian_fn = jax.jacobian(probability_ensemble, argnums=(0, 1))
d_probs_d_t1, d_probs_d_t2 = jacobian_fn(t1, t2, cliffords)
avg_d_probs_d_t1 = jnp.mean(d_probs_d_t1, axis=1)
avg_d_probs_d_t2 = jnp.mean(d_probs_d_t2, axis=1)


print("Sensitivity of '0' probability to noise parameters:")
print(f"{'Depth':>6s} | {'dP/dT1':>12s} | {'dP/dT2':>12s}")
print("-" * 36)
for i, d in enumerate(depths):
    print(f"{d:6d} | {avg_d_probs_d_t1[i]:12.4f} | {avg_d_probs_d_t2[i]:12.4f}")

### Fitting a model to data

Another thing we can use the gradenits to do is fit models. Here, we will produce a dataset using some random noise parameters and sample outcome, as if we were doing a real experiment.

We can then fit our noise model the sampled probabilities and see if we recover the result.

For this cell, we'll need `optax`. If it's not present in the environment, we'll pip install it.

#### Generate sampled data

First, we'll generate sampled data with a 'true' t1 and t2.

In [ ]:
try:
    import optax
except ModuleNotFoundError:
    !pip install optax
    import optax

# --- Step 1: Generate synthetic "experimental" data ---
# Use slightly different noise parameters as the "true" values
true_t1 = 25  # us
true_t2 = 30  # us

true_probs = probability_ensemble(true_t1, true_t2, cliffords)

# Add some shot noise to simulate a real experiment (1000 shots per circuit)
num_shots = 1000
key_sample = jax.random.key(42)
noisy_counts = jax.random.binomial(key_sample, n=num_shots, p=true_probs)
sampled_probs = noisy_counts / num_shots
uncertainties = jnp.sqrt(sampled_probs * (1 - sampled_probs) / num_shots)

print("Synthetic experimental data (with shot noise):")
print(f"{'Depth':>6s} | {'True (mean)':>12s} | {'Sampled (mean)':>14s} | {'Uncertainty (mean)':>18s}")
print("-" * 58)
for i, d in enumerate(depths):
    print(f"{d:6d} | {true_probs[i].mean():12.4f} | {sampled_probs[i].mean():14.4f} | {uncertainties[i].mean():18.4f}")

Here, we don't want to merely fit to the average probability of 0, but to the specific measured probabilities of each sequence. Sequences may have different sensivities to the errors.

In [ ]:
# --- Step 2: Define the loss function and fit using optax ---
@jax.jit
def loss_fn(params):
    """Mean squared error between predicted and sampled probabilities across
    all depths and randomizations."""
    t1_param, t2_param = params
    predicted = probability_ensemble(t1_param, t2_param, cliffords)
    return jnp.mean((predicted - sampled_probs) ** 2)


grad_fn = jax.jit(jax.grad(loss_fn))

# Initial guess in µs (deliberately offset from truth)
initial_t1, initial_t2 = 35.0, 40.0
params = jnp.array([initial_t1, initial_t2])

# Set up an Adam optimizer with optax
optimizer = optax.adam(learning_rate=0.1)
opt_state = optimizer.init(params)

# Run the optimization loop
num_steps = 300
for step in range(num_steps):
    loss = loss_fn(params)
    grads = grad_fn(params)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    if step % 50 == 0 or step == num_steps - 1:
        print(f"Step {step:4d} | Loss = {loss:.2e} | T1 = {params[0]:.2f} µs | T2 = {params[1]:.2f} µs")

fitted_t1, fitted_t2 = params
print("\n--- Fitting Results ---")
print(f"Initial  T1 = {initial_t1:.2f} µs,  T2 = {initial_t2:.2f} µs")
print(f"Fitted   T1 = {fitted_t1:.2f} µs,  T2 = {fitted_t2:.2f} µs")
print(f"True     T1 = {true_t1:.2f} µs,  T2 = {true_t2:.2f} µs")

## Interleaved Randomized Benchmarking (IRB)

Next, we'll do the same experiment with interleaved randomized benchmarking. This case is slightly more complicated as we'll be using both 1Q and 2Q gates.